# H3 · Verdict head v0 (Himasri)

Runs H3 on Colab: zero-shot NLI scores for every MultiFC snippet (GPU, ~20 min on a T4),
then trains the verdict head v0 (no twin loss) with seeds 42/43/44 and pushes the outputs.

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**.
2. Colab Secrets (key icon on the left), with notebook access switched on:
   - `GH_TOKEN`: GitHub token with write access to `himaallu/Group-Project` (needed for the push at the end).
   - `WANDB_KEY` (optional): if missing, runs are logged offline and you download them at the end to sync later.
3. Run the cells top to bottom. Clear outputs before pushing the notebook itself.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/himaallu/Group-Project.git /content/twincheck
%cd /content/twincheck
!git checkout -q himasri/H3-verdict-head
# Rathi's R2 files (until R2 is merged into main); they are NOT committed from here.
!git fetch -q origin rathi/R2-reranker
!git checkout origin/rathi/R2-reranker -- features/relevance_en.parquet features/entities_en.jsonl
!git reset -q features/   # keep them out of the next commit
!pip install -q -r requirements.txt
!ls -la data/processed/en features

## 2. W&B (logs every training run to project `twincheck`)

In [ ]:
import os
from google.colab import userdata
try:
    key = userdata.get('WANDB_KEY')
except Exception:
    key = None
if key:
    import wandb
    wandb.login(key=key)
    print('W&B: online')
else:
    os.environ['WANDB_MODE'] = 'offline'   # runs saved to ./wandb, sync later with `wandb sync`
    print('W&B: offline (no WANDB_KEY secret) - download wandb_runs.zip at the end')

## 3. 10-row check first (CLAUDE.md rule 3)
Outputs go to `/content/test10`, not into the repo.

In [ ]:
!python -m src.nli.score_zeroshot data/processed/en/multifc_train.jsonl data/processed/en/multifc_val.jsonl data/processed/en/multifc_test.jsonl --limit 10 --out-dir /content/test10/nli
!python -m src.verdict.train --name test10 --limit 10 --no-wandb --epochs 3 --batch-groups 4 \
    --nli /content/test10/nli/multifc_train_first10.parquet /content/test10/nli/multifc_val_first10.parquet /content/test10/nli/multifc_test_first10.parquet \
    --results-dir /content/test10/results --models-dir /content/test10/models | tail -5
!ls /content/test10/results

## 4. Zero-shot NLI on all MultiFC snippets (~154k pairs, ~20 min on a T4)

In [ ]:
!python -m src.nli.score_zeroshot data/processed/en/multifc_val.jsonl data/processed/en/multifc_test.jsonl data/processed/en/multifc_train.jsonl --batch-size 64
!ls -la features/nli_zeroshot

## 5. Train v0 (λ_inv = λ_sens = 0), seeds 42 / 43 / 44

In [ ]:
NLI = 'features/nli_zeroshot/multifc_train.parquet features/nli_zeroshot/multifc_val.parquet features/nli_zeroshot/multifc_test.parquet'
for seed in (42, 43, 44):
    !python -m src.verdict.train --name v0 --seed {seed} --nli {NLI} | tail -25

## 6. Summary (paste this into the chat with Claude)

In [ ]:
import json, pandas as pd
rows = []
for seed in (42, 43, 44):
    m = json.load(open(f'results/H3/metrics_v0_s{seed}.json'))
    rows.append({'seed': seed, 'best_epoch': m['config']['best_epoch'],
                 **{f'val_{k}': m['val'][k] for k in ('macro_f1', 'auroc', 'ece')},
                 **{f'test_{k}': m['test'][k] for k in ('macro_f1', 'auroc', 'ece')}})
df = pd.DataFrame(rows).set_index('seed')
print(df.round(4).to_string())
print('\nmean:'); print(df.mean().round(4).to_string())
print('sd:');   print(df.std(ddof=1).round(4).to_string())

## 7. Push the outputs to `himasri/H3-verdict-head`
Fill in your name and GitHub email first. R2 files and `wandb/` are not pushed.

In [ ]:
GIT_NAME = 'Himasri'
GIT_EMAIL = 'YOUR_GITHUB_EMAIL'
token = userdata.get('GH_TOKEN')
!git config user.name "{GIT_NAME}"
!git config user.email "{GIT_EMAIL}"
!git add features/nli_zeroshot results/H3 models/verdict_v0_s42 models/verdict_v0_s43 models/verdict_v0_s44
!git status --short
!git commit -q -m "[H3] Zero-shot NLI features and v0 verdict head (seeds 42/43/44)"
!git push -q https://{token}@github.com/himaallu/Group-Project.git himasri/H3-verdict-head && echo pushed

## 8. Only if W&B was offline: download the runs to sync later

In [ ]:
import os
if os.environ.get('WANDB_MODE') == 'offline':
    !zip -qr /content/wandb_runs.zip wandb
    from google.colab import files
    files.download('/content/wandb_runs.zip')
    # later, on any machine with W&B set up:  unzip wandb_runs.zip && wandb sync wandb/offline-run-*

When done: Runtime → Disconnect and delete runtime (frees your GPU units).